# ***Librerias***

In [76]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import chi2, zscore
from sklearn.ensemble import IsolationForest

# ***Parametrós***

In [77]:
AQI_BREAKPOINTS = {
            "O3": [(0.000, 0), (0.055, 51), (0.071, 101), (0.086, 151), (0.106, 201), (0.201, 301)],
            "PM2.5": [(0, 0), (9.1, 51), (35.5, 101), (55.5, 151), (125.5, 201), (225.5, 301)],
            "PM10": [(0, 0), (55, 51), (155, 101), (255, 151), (255, 201), (425, 301)],
            "CO": [(0.0, 0), (4.5, 51), (9.5, 101), (12.5, 151), (15.5, 210), (30.5, 301)],
            "SO2": [(0.000, 0), (0.036, 51), (0.076, 101), (0.186, 151), (0.305, 201), (0.605, 301)],
            "NO2": [(0.000, 0), (0.054, 51), (0.101, 101), (0.360, 151), (0.650, 201), (1.250, 301)]
        }

# ***Funciones***

In [79]:
"""Limpieza"""

def limpiar_gas_profundo(serie):

    if serie.dropna().empty: return serie
    s = serie.dropna()

    # IQR
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    s = s[(s >= q1 - 1.5*iqr) & (s <= q3 + 1.5*iqr)]

    # Mahalanobis
    if len(s) > 2:
        mu, var = s.mean(), s.var() + 1e-6
        dist = ((s - mu)**2) / var
        s = s[dist <= chi2.ppf(0.99, df=1)]
        # Z-score
        s = s[np.abs(zscore(s)) < 3]

    return serie.where(serie.index.isin(s.index))

In [80]:
"""Limpieza con Isolation Forest"""

def limpiar_pm_profundo(serie):

    if serie.dropna().empty: return serie
    s = serie.dropna().to_frame()
    model = IsolationForest(contamination=0.05, random_state=42)
    preds = model.fit_predict(s)
    return serie.where(pd.Series(preds == 1, index=s.index))

In [81]:
"""Limpieza IQR para meteorología"""

def limpiar_meteo_iqr(serie):

    q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
    iqr = q3 - q1
    return serie.mask((serie < q1 - 1.5*iqr) | (serie > q3 + 1.5*iqr))

In [82]:
""" Calcular AQI"""

def calcular_aqi_valor(conc, cont):
    pts = AQI_BREAKPOINTS.get(cont)
    if not pts or pd.isna(conc): return np.nan
    for i in range(len(pts)-1):
        (c_l, i_l), (c_h, i_h) = pts[i], pts[i+1]
        if c_l <= conc <= c_h:
            return round(((i_h - i_l)/(c_h - c_l)) * (conc - c_l) + i_l, 2)
    return np.nan

In [83]:
"""Ejecuta toda la limpieza y cálculo de ICA para una estación específica."""

def procesar_estacion(df_raw, nombre):
    df = df_raw.copy()

    # SOLUCIÓN ERROR 'ND': Convertir a numérico antes de cualquier operación
    cols_datos = [c for c in df.columns if c not in ['Fecha', 'Hora']]
    for col in cols_datos:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Identificar variables presentes para evitar el 'KeyError'
    contaminantes = [c for c in ['PM10', 'PM2.5', 'CO', 'O3', 'NO2', 'SO2'] if c in df.columns]
    meteorologia = [c for c in ['TM', 'HR', 'VV', 'DV'] if c in df.columns]

    # Limpieza profunda (Tus métodos originales)
    for c in contaminantes:
        df[c] = limpiar_pm_profundo(df[c]) if 'PM' in c else limpiar_gas_profundo(df[c])
        # Conversión dinámica ppb -> ppm
        if c in ['O3', 'NO2', 'SO2']: df[f'{c}_ppm'] = df[c] * 0.001
        elif c == 'CO': df[f'{c}_ppm'] = df[c]

    for m in meteorologia:
        df[m] = limpiar_meteo_iqr(df[m])

    # Resample e Interpolación (Frecuencia 'h')
    df['Fecha'] = pd.to_datetime(df['Fecha'])
    df = df.set_index('Fecha').resample('h').mean().interpolate(method='time').ffill().bfill()

    # Cálculo de ICA (AQI) con redondeo a 2 decimales
    res = pd.DataFrame(index=df.index)
    aqi_cols = []

    # Mapeo de ventanas de tiempo EPA
    config = {'O3_ppm':('8h','O3'), 'CO_ppm':('8h','CO'), 'PM10':('24h','PM10'), 'PM2.5':('24h','PM2.5')}

    for col, (window, label) in config.items():
        if col in df.columns:
            serie = df[col].rolling(window).mean()
            res[f'ICA_{label}'] = serie.apply(lambda x: calcular_aqi_valor(x, label))
            aqi_cols.append(f'ICA_{label}')

    # ICA Final y Meteorología
    res['ICA_Target'] = res[aqi_cols].max(axis=1).round(2)
    for m in meteorologia:
        res[m] = df[m].round(2)

    return res.dropna().reset_index()

In [84]:
def ejecutar_limpieza_total(archivo):
    print(f"Abriendo: {archivo}")
    hojas = pd.read_excel(archivo, sheet_name=None)

    for nombre in hojas.keys():
        print(f"Procesando: {nombre}...")
        try:
            # Se pasan exactamente 2 argumentos: el dataframe y el nombre
            df_final = procesar_estacion(hojas[nombre], nombre)
            df_final.to_csv(f"Datos_Procesados_{nombre}_Validacion.csv", index=False)
        except Exception as e:
            print(f"-> Error en {nombre}: {e}")

# ***Procesamiento***

In [85]:
ejecutar_limpieza_total('Datos_estaciones_Validacion.xlsx')

Abriendo: Datos_estaciones_Validacion.xlsx
Procesando: INFO...
-> Error en INFO: 'Fecha'
Procesando: CAP...
Procesando: EPG...
Procesando: FEO...
